# Exercise 3 Pandas

This is an implementation document for the third Python exercise. The objective is to create a clear report with the code implementation and the explanation of the object-oriented design and data analysis workflow.

Before the System Design & Implementation section, the required libraries are imported to enable data handling with pandas and perform numerical computations using numpy. 

In [119]:
import pandas as pd
import numpy as np

## 1. System Design & Implementation

The `CovidDataAnalyzer` class represents the main entity of the exercise and is responsible for loading, cleaning, filtering, analyzing and exporting COVID-19 data from the given `covid_19_data.csv` file. 

Its attributes are: `file_path`, which stores the path of the CSV file, `data`, which stores the full dataset after loading it with pandas, and `filtered_data`, which stores subsets of the original dataset after the application of filters. The same object keeps both the original data and the intermediate results, which can lead to a more organized workflow.

The constructor `(__init__)` initializes these attributes in a simple way: the file path is provided by the user when the object is created, while data and filtered_data are initialized as None until real DataFrame objects are assigned to them during execution. The data loading is handled in a separate load_data() method to maintain clear separation of responsibilities within the class. This approach makes the state of the object explicit and prevents uncertainty about whether data has already been loaded or filtered. It is important to use the full file path in the `CovidDataAnalyzer` method to access the file correctly. 

In [120]:
class CovidDataAnalyzer:
    """
    Represents a COVID-19 data analyzer.

    Attributes:
        file_path (str): The path to the CSV file containing the dataset.
        data (pd.DataFrame | None): The full dataset loaded from the CSV file.
        filtered_data (pd.DataFrame | None): A subset of the dataset after applying filters.
    """
    def __init__(self, file_path: str) -> None:
        """
        Initialize a CovidDataAnalyzer object.

        Args:
            file_path (str): The path to the CSV file containing the COVID-19 dataset.

        Returns:
            None
        """
        self.file_path = file_path
        self.data = None
        self.filtered_data = None

    def load_data(self) -> pd.DataFrame:
        """
        Load the dataset from the CSV file and store it in the data attribute.

        Args:
            None

        Returns:
            pd.DataFrame: The loaded dataset.
        """
        self.data = pd.read_csv(self.file_path)

        print("Data loaded successfully.")
        return self.data

    def describe_data(self) -> None:
        """
        Print the main descriptive information of the dataset and provide short insights.

        Args:
            None

        Returns:
            None
        """
        if self.data is None:
            print("No data loaded yet. Please run load_data() first.")
            return
        
        print('\n---------------------------------------------------------------------------------------------------\n')
        print("SHAPE (rows, columns)")
        print(self.data.shape)

        print('\n---------------------------------------------------------------------------------------------------\n')
        print("COLUMN NAMES")
        print(self.data.columns.tolist())

        print('\n---------------------------------------------------------------------------------------------------\n')
        print("DATA TYPES")
        print(self.data.dtypes)

        print('\n---------------------------------------------------------------------------------------------------\n')
        print("FIRST 5 ROWS")
        print(self.data.head(5))

        print('\n---------------------------------------------------------------------------------------------------\n')
        print("MISSING VALUES")
        print(self.data.isnull().sum())

        print('\n---------------------------------------------------------------------------------------------------\n')
        print("BASIC STATISTICS")
        print(self.data.describe(include="all"))

    def handle_missing_values(self) -> None:
        """
        Handle missing values in the dataset.

        Args:
            None

        Returns:
            None
        """
        if self.data is None:
            print("No data loaded yet. Please run load_data() first.")
            return

        numeric_columns = self.data.select_dtypes(include=[np.number]).columns
        categorical_columns = self.data.select_dtypes(exclude=[np.number]).columns

        self.data[numeric_columns] = self.data[numeric_columns].fillna(0)
        self.data[categorical_columns] = self.data[categorical_columns].fillna("Unknown")

        print("Missing values handled successfully.")
        
    def filter_high_cases(self) -> pd.DataFrame:
        """
        Filter the dataset using multiple epidemiological conditions.
        
        Args:
            None

        Returns:
            pd.DataFrame: The filtered dataset.
        """
        if self.data is None:
            print("No data loaded yet. Please run load_data() first.")
            return pd.DataFrame()

        self.filtered_data = self.data[
            (self.data["Confirmed"] > 100000) &
            (self.data["Deaths"] > 5000) &
            (self.data["Country/Region"] != "Unknown")
        ]

        print("High cases filter applied successfully.")
        return self.filtered_data

    def filter_by_date_range(self, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Filter the dataset within a specific date range.

        Args:
            start_date (str): The starting date of the interval.
            end_date (str): The ending date of the interval.

        Returns:
            pd.DataFrame: The filtered dataset for the given date range.
        """
        if self.data is None:
            print("No data loaded yet. Please run load_data() first.")
            return pd.DataFrame()

        if "Date" not in self.data.columns:
            print("The dataset does not contain a 'Date' column.")
            return pd.DataFrame()
        else:
            self.data["Date"] = pd.to_datetime(self.data["Date"], errors="coerce")
        
        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)

        self.filtered_data = self.data[
            (self.data["Date"] >= start_date) &
            (self.data["Date"] <= end_date)
        ]

        print("Date range filter applied successfully.")
        return self.filtered_data

    def calculate_global_statistics(self) -> None:
        """
        Calculate and print the global totals of confirmed, deaths and recovered cases.

        Args:
            None

        Returns:
            None
        """
        if self.data is None:
            print("No data loaded yet. Please run load_data() first.")
            return

        total_confirmed = self.data["Confirmed"].sum()
        total_deaths = self.data["Deaths"].sum()
        total_recovered = self.data["Recovered"].sum()

        print("Global Statistics:")
        print(f"Total Confirmed Cases: {total_confirmed:,}")
        print(f"Total Deaths: {total_deaths:,}")
        print(f"Total Recovered Cases: {total_recovered:,}")

    def save_filtered_data(self, filename: str) -> None:
        """
        Save the filtered dataset to a CSV file.
        
        Args:
            filename (str): The name of the CSV file to be created.

        Returns:
            None
        """
        if self.filtered_data is None:
            print("No filtered data available to save.")
            return

        self.filtered_data.to_csv(filename, index=False)
        print(f"Filtered data saved successfully to {filename}.")

The `load_data` and `describe_data` methods are the first stage of the analysis workflow. The dataset is imported with the `pandas.read_csv` command, which transforms the CSV file into a DataFrame that can be efficiently manipulated. In the first, the `Date` column is converted to datetime format so that date-based filtering can be performed correctly and this format ensures that later comparisons involving dates will be valid and reliable. The second displays the dataset dimensions (shape), available variables (columns), data types, first rows, missing values and general descriptive statistics in order to give an overview of the structure and quality of the data. In the specific dataset used, special attention should be given to the `Province/State` column, because it contains many missing values, while the `Country/Region` and epidemiological columns are complete and therefore suitable for global analysis.

The `handle_missing_values` method improves data quality by replacing null values according to the column type. Missing values in numeric columns are replaced with `0`, which allows arithmetic operations such as sums and comparisons to proceed without interruption. Missing values in non-numeric columns are replaced with the string `Unknown`, which prevents missing category values from remaining blank and makes the dataset easier to interpret.

The `filter_high_cases` and `filter_by_date_range` methods represent two distinct but complementary analytical strategies applied to the dataset. The first performs conditional filtering based on epidemiological severity, by keeping only the rows where confirmed cases are greater than 100000, deaths are greater than 5000 and the country/region is not equal to "Unknown". The second restricts the dataset to a selected time interval, by converting the input dates to datetime format, using pd.to_datetime() with the errors="coerce" parameter which ensures that any invalid or improperly formatted date values are safely converted to missing values, and then uses a pandas comparison to keep only the rows whose Date value falls within the selected interval. This can be useful, especially when studying the evolution of the pandemic during a specified period. Both methods store their output in `filtered_data`, which makes it easy to reuse or export the latest result. 

The `calculate_global_statistics` method sums the most important quantitative indicators in the dataset by calculating the global totals for confirmed, death and recovered cases. This provides a simple but effective summary of the scale of the data under examination.

Finally, the `save_filtered_data` method allows the analyzed subsets to be exported as CSV files. This separates analysis and storage, while also making the results available for later use. The csv file generated is saved in the same folder path as this notebook.

Overall, the class follows a clear object-oriented structure: data and behavior are combined in a single entity, and each method performs one well defined task.

### INSIGHTS
This section outlines the main insights derived from executing the `describe_data` method. Based on the descriptive statistics that are presented in detail in the next section, the following insights can be outlined:
* The dataset contains 49,068 rows and 10 columns, which shows a large collection of COVID-19 records across multiple dates and regions.
* The dataset contains 49,068 observations for the Date column, confirming that there are no missing values in the temporal dimension and that the dataset provides continuous daily tracking.
* The Date column includes 188 unique dates, indicating that the dataset spans a period of several months and allows for time-based analysis of the pandemic’s progression.
* The most frequent date appears 261 times, suggesting that multiple records are reported per day, likely corresponding to different countries or regions.
* The presence of NaN values in statistical measures such as mean and standard deviation for the Date column indicates that these metrics are not applicable to datetime data, rather than reflecting missing or incomplete data.
* The Confirmed cases show a very large range (from 0 to over 4 million (4.290259e+06 in max value)), highlighting the significant variation in COVID-19 impact across different regions and time periods.
* The high standard deviation in Confirmed, Deaths, and Recovered cases suggests that the data is highly dispersed, which means that case numbers differ greatly between observations.
* The median values (50%) for deaths and recovered cases are relatively low compared to the maximum values, which indicatea that most records correspond to lower case counts, while a smaller number of records contain extremely high values.

## 2. Application Usage & Execution Examples

This section presents the practical implementation and execution of the developed system through a series of example operations. It demonstrates how the defined class interacts with the provided COVID-19 dataset in a realistic analysis scenario, including loading the data, cleaning it, filtering it according to epidemiological and temporal criteria, exporting intermediate results and calculating global summary statistics. The following code illustrates the dynamic behavior of the implemented class and validates the functionality of the required methods.

It is important to use the full file path in the `CovidDataAnalyzer` method to access the file correctly.

In [121]:
# Create a CovidDataAnalyzer object
analyzer = CovidDataAnalyzer(r"C:\Users\despo\Documents\AI Data Factory\Python\Lectures\Snippets\covid_19_data.csv")

In [122]:
# 1. Load and describe the dataset
analyzer.load_data()
analyzer.describe_data()

Data loaded successfully.

---------------------------------------------------------------------------------------------------

SHAPE (rows, columns)
(49068, 10)

---------------------------------------------------------------------------------------------------

COLUMN NAMES
['Province/State', 'Country/Region', 'Lat', 'Long', 'Date', 'Confirmed', 'Deaths', 'Recovered', 'Active', 'WHO Region']

---------------------------------------------------------------------------------------------------

DATA TYPES
Province/State        str
Country/Region        str
Lat               float64
Long              float64
Date                  str
Confirmed           int64
Deaths              int64
Recovered           int64
Active              int64
WHO Region            str
dtype: object

---------------------------------------------------------------------------------------------------

FIRST 5 ROWS
  Province/State Country/Region       Lat       Long        Date  Confirmed  \
0            NaN    Af

In [123]:
# 2. Handle missing values
analyzer.handle_missing_values()

Missing values handled successfully.


In [124]:
# 3. Apply the high-cases filter and save the result
high_cases_data = analyzer.filter_high_cases()
print(high_cases_data.head(5))
analyzer.save_filtered_data("high_cases_filtered.csv")

High cases filter applied successfully.
      Province/State Country/Region        Lat       Long        Date  \
17883        Unknown          Italy  41.871940   12.56738  2020-03-30   
18144        Unknown          Italy  41.871940   12.56738  2020-03-31   
18232        Unknown             US  40.000000 -100.00000  2020-03-31   
18405        Unknown          Italy  41.871940   12.56738  2020-04-01   
18469        Unknown          Spain  40.463667   -3.74922  2020-04-01   

       Confirmed  Deaths  Recovered  Active WHO Region  
17883     101739   11591      14620   75528     Europe  
18144     105792   12428      15729   77635     Europe  
18232     188724    5605       7024  176095   Americas  
18405     110574   13155      16847   80572     Europe  
18469     104118    9387      22647   72084     Europe  
Filtered data saved successfully to high_cases_filtered.csv.


In [125]:
# 4. Apply the date-range filter for March 2020 to June 2020 and save the result
date_filtered_data = analyzer.filter_by_date_range("2020-03-01", "2020-06-30")
print(date_filtered_data.head(5))
analyzer.save_filtered_data("date_range_filtered.csv")

Date range filter applied successfully.
      Province/State Country/Region       Lat       Long       Date  \
10179        Unknown    Afghanistan  33.93911  67.709953 2020-03-01   
10180        Unknown        Albania  41.15330  20.168300 2020-03-01   
10181        Unknown        Algeria  28.03390   1.659600 2020-03-01   
10182        Unknown        Andorra  42.50630   1.521800 2020-03-01   
10183        Unknown         Angola -11.20270  17.873900 2020-03-01   

       Confirmed  Deaths  Recovered  Active             WHO Region  
10179          1       0          0       1  Eastern Mediterranean  
10180          0       0          0       0                 Europe  
10181          1       0          0       1                 Africa  
10182          0       0          0       0                 Europe  
10183          0       0          0       0                 Africa  
Filtered data saved successfully to date_range_filtered.csv.


In [126]:
# 5. Calculate and display global statistics
analyzer.calculate_global_statistics()

Global Statistics:
Total Confirmed Cases: 828,508,482
Total Deaths: 43,384,903
Total Recovered Cases: 388,408,229
